# Reshape & Merge Dataset

**Objective:** Convert all 5 hourly wide-format CSVs (time × 275 zones) into a single long-format DataFrame, then join zone and station metadata.

**Input:** Raw CSVs from `data/raw/`  
**Output:** `data/processed/merged_hourly_data.csv` - one row per (time, zone_id) with columns for occupancy, volume, duration, e_price, s_price, and zone features

## Import Required Libraries


In [1]:
import pandas as pd
import numpy as np
import gc
import time as timer
from pathlib import Path

print("Libraries loaded!")

Libraries loaded!


## 1. Load Raw Data

### 1.1 Load Dataset & Melt Function

In [2]:
DATA_DIR = Path("../data/raw")
HOURLY = DATA_DIR / "charge_1hour"

def load_and_melt(csv_path, value_name):
    """
    Load a wide-format CSV (time × 275 zones) and reshape to long format.
    Returns DataFrame with columns: [time, zone_id, <value_name>]
    """
    df = pd.read_csv(csv_path)
    df.rename(columns={df.columns[0]: "time"}, inplace=True)
    df["time"] = pd.to_datetime(df["time"])
    
    zone_cols = [c for c in df.columns if c != "time"]
    melted = df.melt(
        id_vars="time",
        value_vars=zone_cols,
        var_name="zone_id",
        value_name=value_name,
    )

    melted["zone_id"] = melted["zone_id"].astype(np.int16)
    melted[value_name] = melted[value_name].astype(np.float32)

    return melted

### 1.2 File Maping and Loading

In [3]:
FILE_MAP = {
    "occupancy": "occupancy.csv",
    "volume":    "volume.csv",
    "duration":  "duration.csv",
    "e_price":   "e_price.csv",
    "s_price":   "s_price.csv",
}

loaded = {}

for name, fname in FILE_MAP.items():
    loaded[name] = load_and_melt(HOURLY / fname, name)

print(f"\nAll files melted and loaded!")


All files melted and loaded!


## 2. Merge All Hourly Datasets

Merge the five DataFrames on `(time, zone_id)` to create a single DataFrame with one row per zone per hour.

In [4]:
keys = ["time", "zone_id"]
df = loaded["occupancy"].copy()

for name in ["volume", "duration", "e_price", "s_price"]:
    df = df.merge(loaded[name], on=keys, how="left")

# Free memory
del loaded
gc.collect()

print(f"Merged shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Merged shape: (1194600, 7)
Columns: ['time', 'zone_id', 'occupancy', 'volume', 'duration', 'e_price', 's_price']


,time,zone_id,occupancy,volume,duration,e_price,s_price
0,2022-09-01 00:00:00,102,17.0,56.291668,8.833333,0.924,0.856
1,2022-09-01 01:00:00,102,16.0,50.458332,8.166667,0.924,0.856
2,2022-09-01 02:00:00,102,16.0,55.708332,8.916667,0.924,0.856
3,2022-09-01 03:00:00,102,16.0,57.750000,9.166667,0.924,0.856
4,2022-09-01 04:00:00,102,17.0,58.333332,9.250000,0.924,0.856


## 3. Join Zone Metadata

Add zone-level spatial features: longitude, latitude, charge_count, area, perimeter.

In [5]:
zone_info = pd.read_csv(DATA_DIR / "zone-information.csv")
zone_info.rename(columns={"TAZID": "zone_id"}, inplace=True)

df = df.merge(zone_info, on="zone_id", how="left")

print(f"After zone merge: {df.shape}")
print(f"Zone columns added: {zone_info.columns.tolist()}")

After zone merge: (1194600, 12)
Zone columns added: ['zone_id', 'longitude', 'latitude', 'charge_count', 'area', 'perimeter']


## 4. Aggregate Station Information Per Zone

Compute per-zone station statistics: number of stations, total piles, mean coordinates.

In [6]:
si = pd.read_csv(DATA_DIR / "station_information.csv")
si.rename(columns={"TAZID": "zone_id"}, inplace=True)

station_agg = (
    si.groupby("zone_id")
    .agg(
        num_stations=("station_id", "nunique"),
        total_piles=("charge_count", "sum"),
        mean_station_lat=("latitude", "mean"),
        mean_station_lon=("longitude", "mean"),
    )
    .reset_index()
)

df = df.merge(station_agg, on="zone_id", how="left")

print(f"After station merge: {df.shape}")
print(f"Station columns added: {station_agg.columns.tolist()}")

After station merge: (1194600, 16)
Station columns added: ['zone_id', 'num_stations', 'total_piles', 'mean_station_lat', 'mean_station_lon']


## 5. Data Quality Check

In [7]:
print(f"Final shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Date range: {df['time'].min()} → {df['time'].max()}")
print(f"Zones: {df['zone_id'].nunique()}")
print(f"Timestamps: {df['time'].nunique()}")

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"Count": missing, "%": missing_pct})

if missing_report[missing_report["Count"] > 0].empty:
    print("No missing values!")

Final shape: (1194600, 16)
Memory: 121.8 MB
Date range: 2022-09-01 00:00:00 → 2023-02-28 23:00:00
Zones: 275
Timestamps: 4344
No missing values!


In [8]:
print("Final columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:<25s} {str(df[col].dtype)}")

print(f"\nSample rows:")
df.sample(5, random_state=42)

Final columns:
   1. time                      datetime64[ns]
   2. zone_id                   int16
   3. occupancy                 float32
   4. volume                    float32
   5. duration                  float32
   6. e_price                   float32
   7. s_price                   float32
   8. longitude                 float64
   9. latitude                  float64
  10. charge_count              int64
  11. area                      float64
  12. perimeter                 float64
  13. num_stations              int64
  14. total_piles               int64
  15. mean_station_lat          float64
  16. mean_station_lon          float64

Sample rows:


,time,zone_id,occupancy,volume,duration,e_price,s_price,longitude,latitude,charge_count,area,perimeter,num_stations,total_piles,mean_station_lat,mean_station_lon
968180,2023-02-06 20:00:00,1074,6.0,29.166666,4.166667,1.022857,0.700000,113.921388,22.539396,124,9.988738e+05,4162.7267,7,62,22.538033,113.922262
286722,2022-09-01 18:00:00,525,24.0,90.708336,15.333333,1.015714,0.665714,113.877453,22.563339,96,1.721568e+06,5219.8913,7,96,22.562044,113.875996
226172,2022-09-12 20:00:00,348,4.0,28.000000,4.000000,0.700000,0.760000,0.000000,0.000000,132,3.082857e+06,11446.4803,1,33,22.522460,113.950976
1003203,2023-02-18 03:00:00,1088,8.0,28.000000,6.000000,1.010000,0.736000,113.924193,22.512670,290,7.655050e+05,3724.5077,5,69,22.512066,113.924500
288489,2022-11-14 09:00:00,525,21.0,114.625000,18.250000,0.961429,0.774286,113.877453,22.563339,96,1.721568e+06,5219.8913,7,96,22.562044,113.875996


## 6. Save Merged Dataset

In [9]:
# Split dataframe in halfs
split_point = len(df) // 2
df_part1 = df.iloc[:split_point]
df_part2 = df.iloc[split_point:]

output_path1 = "../data/processed/merged_hourly_data_01.csv"
output_path2 = "../data/processed/merged_hourly_data_02.csv"

df_part1.to_csv(output_path1, index=False)
df_part2.to_csv(output_path2, index=False)

print(f"Saved Part 1 to: {output_path1}")
print(f"  Shape: {df_part1.shape}")
print(f"  File size: {Path(output_path1).stat().st_size / 1e6:.1f} MB")

print(f"\nSaved Part 2 to: {output_path2}")
print(f"  Shape: {df_part2.shape}")
print(f"  File size: {Path(output_path2).stat().st_size / 1e6:.1f} MB")

print(f"\nTotal rows saved: {len(df_part1) + len(df_part2)}")

Saved Part 1 to: ../data/processed/merged_hourly_data_01.csv
  Shape: (597300, 16)
  File size: 82.2 MB

Saved Part 2 to: ../data/processed/merged_hourly_data_02.csv
  Shape: (597300, 16)
  File size: 82.3 MB

Total rows saved: 1194600
